### Loading Environment Variables

## To authenticate requests securely, the Anthropic API key is stored outside the source code in a local .env file. This prevents sensitive credentials from being exposed in the project files.

## The python-dotenv package loads the environment variables into the current Python session, while the os module retrieves the value associated with ANTHROPIC_API_KEY. Finally, a boolean check confirms that the API key has been successfully loaded.

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("ANTHROPIC_API_KEY")

print(api_key is not None)

True


In [1]:
%pip install anthropic


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip show anthropic

Name: anthropic
Version: 0.109.2
Summary: The official Python library for the anthropic API
Home-page: https://github.com/anthropics/anthropic-sdk-python
Author: 
Author-email: Anthropic <support@anthropic.com>
License: MIT
Location: /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages
Requires: anyio, distro, docstring-parser, httpx, jiter, pydantic, sniffio, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


### Initializing the Anthropic Client

## The Anthropic SDK provides a client object that serves as the interface between the Python application and the Claude API. Once initialized, the client manages authenticated requests using the API key loaded from the environment variables.

## Printing a confirmation message verifies that the client has been successfully created and is ready to send requests to Claude.

In [3]:
from anthropic import Anthropic

client = Anthropic()

print("Client initialized")

Client initialized


### Building a Multi-turn Conversation

The Claude Messages API is stateless and does not retain conversation history between requests. To simulate a continuous dialogue, a list named `messages` stores every user prompt and every assistant response.

The `ask_claude()` function appends the user's message to the conversation history, sends the complete history to the Claude API, extracts the assistant's response, and stores it for future context. Additionally, the function reports the number of input and output tokens consumed by each request, providing visibility into the model's token usage and supporting more efficient prompt design.


In [5]:
messages = []

def ask_claude(prompt):

    messages.append(
        {
            "role": "user",
            "content": prompt
        }
    )

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=messages
    )

    assistant_text = response.content[0].text

    messages.append(
        {
            "role": "assistant",
            "content": assistant_text
        }
    )

    print("\n--- TOKEN USAGE ---")
    print(f"Input Tokens:  {response.usage.input_tokens}")
    print(f"Output Tokens: {response.usage.output_tokens}")
    print("-------------------\n")

    return assistant_text

### Running an Interactive Chat Session

## The main program runs inside an infinite loop, allowing the user to interact with Claude through the terminal. Each iteration prompts for user input, sends the message to the `ask_claude()` function, and displays the model's response.

## The loop continues until the user enters `quit` or `exit`, providing a simple command-line interface that supports multi-turn conversations while preserving the accumulated conversation history. Informational messages are printed before and after each API request to indicate the progress of the interaction.


In [ ]:
while True:

    user_input = input("You: ")

    if user_input.lower() in ["quit", "exit"]:
        break

    print("Sending request...")

    response = ask_claude(user_input)

    print("Received response.")

    print(response)